## 1 — ライブラリの読込み

- 処理: ライブラリの読込み
- 入力: プロジェクトのPython環境
- 出力: 状態確認・起動用の関数
- 次: 共有JSONを指定

- 表はクリックで展開 / 縦横スクロール / CSVは同じ列順の全行・全精度

- 共通評価は新規発見・継続的な嗜好・多様性／発見性・効率性 / 集計は02、定義は[評価仕様](../docs/evaluation.md)


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

from IPython.display import display

from src.interfaces.cli.lab_data import verify_prepared
from src.interfaces.notebook.notebook_config import ExperimentConfigPicker
from src.interfaces.notebook.notebook_display import show_details, show_state
from src.interfaces.notebook.search_analysis import workflow_state

## 2 — 実験設定を選択

- 処理: configs内のlistening-v2設定を選択し、読み込むボタンで確定
- 入力: 選択欄のJSON / 初期状態は未選択
- 出力: 読み込んだ実験名・モデル・入力パス / 評価は開始しない
- 次: 読込み済みの表示を確認して次セルへ / ファイル追加後は一覧を更新


In [ ]:
ROOT = Path.cwd() if Path("src").exists() else Path.cwd().parent
config_picker = ExperimentConfigPicker(ROOT / "configs")
display(config_picker.widget)

## 3 — 選択した設定の確認

- 処理: 明示的に読み込んだ設定を後続セルで使用
- 入力: 選択欄の読込み済み設定
- 出力: WORKFLOWと実験名 / 未指定なら案内して停止
- 次: 後続セルを上から順に実行 / 別のNotebookでも同じファイルを選択


In [ ]:
WORKFLOW = config_picker.require_loaded()
EXPERIMENT_CONFIG = WORKFLOW.path
print("実験:", WORKFLOW.name)
os.chdir(ROOT)
CONFIG = EXPERIMENT_CONFIG

## 4 — 準備状態の確認

- 処理: 準備状態の確認
- 入力: Validation入力・ready.json
- 出力: 照合結果または未準備の案内
- 次: 準備後は01で入力確認


In [ ]:
config_picker.require_current(globals().get("WORKFLOW"))
BUNDLE = WORKFLOW.validation_dataset.parent
ready_path = BUNDLE / "ready.json"
if ready_path.exists():
    READY = verify_prepared(BUNDLE)
    print(f"入力照合済み / {READY['users']:,}ユーザー / {READY['tracks']:,}曲")
    show_details({k: READY[k] for k in ("snapshot_id", "split_id")}, "入力の識別情報")
else:
    READY = None
    print("準備完了情報なし / 入力の準備状態を確認してください")

## 5 — 評価の手動開始

- 処理: 評価の手動開始
- 入力: ACTION = None / "evaluate"
- 出力: 明示時に別プロセスのPIDとログパス
- 次: 次セルで終了を確認


In [ ]:
config_picker.require_current(globals().get("WORKFLOW"))
ACTION = None  # None / "evaluate"
if ACTION is not None:
    if ACTION != "evaluate" or WORKFLOW.model not in {"popularity", "itemcf"}:
        raise ValueError("全件評価はACTION=evaluateを指定")
    if READY is None:
        raise RuntimeError("データ準備完了後に入力確認セルを再実行してください")
    WORKFLOW.output.mkdir(parents=True, exist_ok=True)
    env = dict(os.environ, OMP_NUM_THREADS="2", OPENBLAS_NUM_THREADS="2", MKL_NUM_THREADS="2")
    job = subprocess.Popen(
        [
            sys.executable,
            "-u",
            "-m",
            "src.interfaces.cli.grid" if WORKFLOW.grid else "src.interfaces.cli.evaluate",
            "--config",
            str(CONFIG),
        ],
        cwd=ROOT,
        env=env,
        start_new_session=True,
    )
    print("開始 / PID:", job.pid, "/ ログ:", WORKFLOW.output / "run.log")
else:
    print("評価は開始していません / CLIで実行した結果は02で確認")

## 6 — ログと状態の確認

- 処理: ログと状態の確認
- 入力: 最新の実行ログ・study.json
- 出力: ログ末尾と次の操作
- 次: 完了後は02で分析


In [ ]:
config_picker.require_current(globals().get("WORKFLOW"))
show_state(workflow_state(WORKFLOW))
log = WORKFLOW.output / "run.log"
if log.exists():
    show_details("\n".join(log.read_text(errors="replace").splitlines()[-20:]), "run.log")